# Triage XLM-R Training on Kaggle (Dataset mode)

**Prereq:** `bitirme_triage_dataset.zip` dosyasini Kaggle'a **Dataset** olarak yukle:
1. https://www.kaggle.com/datasets → **New Dataset**
2. ZIP'i surukle-birak, title: `bitirme-triage-dataset`
3. Kaggle ZIP'i otomatik acar; dosyalar `/kaggle/input/bitirme-triage-dataset/BitirmeProject/` altinda gorunur
4. Notebook'ta sag panel → **Add Data** → dataset'i ekle

**Notebook settings:**
- Accelerator: **`GPU T4 x2`** (P100 KULLANMA — Kaggle'daki guncel PyTorch P100'u artik desteklemiyor, CUDA sm_60 hatasi alirsin)
- Internet: `ON` (xlm-roberta-base icin HuggingFace'den model indirilecek)

In [ ]:
import pathlib, shutil, torch

INPUT_ROOT = pathlib.Path("/kaggle/input")
PROJECT_DIR = pathlib.Path("/kaggle/working/BitirmeProject")

print("=== /kaggle/input altindaki her sey ===")
for p in sorted(INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []:
    print(" DATASET:", p.name)
    for sub in sorted(p.iterdir())[:10]:
        print("   ", sub.name + ("/" if sub.is_dir() else ""))

def _find_project_root(root: pathlib.Path) -> pathlib.Path | None:
    if not root.exists():
        return None
    for candidate in root.rglob("scripts/train_triage_xlmr.py"):
        return candidate.parent.parent
    return None

DATASET_SRC = _find_project_root(INPUT_ROOT)
assert DATASET_SRC is not None, (
    "train_triage_xlmr.py bulunamadi! /kaggle/input/ altinda dataseti goremiyorsan:\n"
    "  1) Sag panel -> '+ Add Data' -> Your Datasets sekmesinden datasetini ekle\n"
    "  2) Zip'i acilmis mi? 'bitirme_triage_dataset.zip' dosyasini dataset olarak yukleyince\n"
    "     Kaggle otomatik acar; acmadiysan dataseti silip tekrar yukle."
)
print(f"\nProject root tespit edildi: {DATASET_SRC}")

print("\nTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Dataset'i writable dizine kopyala

`/kaggle/input/...` read-only; scriptler `output/out_dataset/` ve `out_models/` altina yazacak.

In [ ]:
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
shutil.copytree(DATASET_SRC, PROJECT_DIR)
print("Copied to:", PROJECT_DIR)
!ls {PROJECT_DIR}

In [ ]:
!pip install -q -U 'transformers>=4.40' 'accelerate>=0.30' 'pyarrow>=14' 'sentencepiece'
import transformers; print('transformers:', transformers.__version__)

## Parquet'leri uret (~10 sn)

In [ ]:
%cd /kaggle/working/BitirmeProject
!python scripts/merge_train_dataset.py
!python scripts/prepare_triage_history_dataset.py
!ls output/out_dataset/

## Egitim (GPU'da ~30-45 dk)

In [ ]:
!python scripts/train_triage_xlmr.py \
    --batch-size 16 \
    --max-len 384 \
    --epochs 4 \
    --lr 2e-5 \
    --label-smoothing 0.05 \
    --patience 2

## Modeli /kaggle/working'e tasi ve tar.gz olarak paketle

In [ ]:
!mkdir -p /kaggle/working/triage_xlmr_out
!cp -r /kaggle/working/BitirmeProject/out_models/triage_xlmr/* /kaggle/working/triage_xlmr_out/
!du -sh /kaggle/working/triage_xlmr_out/
!cd /kaggle/working && tar czf triage_xlmr.tar.gz -C triage_xlmr_out .
!ls -lh /kaggle/working/triage_xlmr.tar.gz

## Bittikten sonra
1. Sag ust **Save Version → Save & Run All (Commit)**
2. Commit bittiginde notebook **Output** sekmesinden `triage_xlmr.tar.gz` indir
3. Local:
   ```bash
   mkdir -p out_models/triage_xlmr
   tar xzf ~/Downloads/triage_xlmr.tar.gz -C out_models/triage_xlmr/
   ```